# 🧠 Area Under the ROC Curve (AUC)

Welcome to the hands-on explanation notebook for **AUC**! In this notebook, we will:
1. Define the mathematical and probabilistic interpretations of AUC.
2. Implement **two different methods** to calculate AUC from scratch:
   - **Method 1 (Trapezoidal Rule Integration):** Integrating the ROC curve coordinates using trapezoids.
   - **Method 2 (Probabilistic Verification):** Performing pairwise comparisons between positive and negative classes to compute:
     $$P(f(x^+) > f(x^-))$$
3. Generate class probability distributions and visualize how class overlap directly affects the AUC.
4. Verify our scratch implementations against `scikit-learn`.
5. Connect AUC to **mean Average Precision (mAP)** in object detection.

Let's start by importing the necessary libraries.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import roc_curve, roc_auc_score

# Set seed for reproducibility
np.random.seed(42)

## 1. Class Distributions and AUC
We will generate three sets of predictions representing different levels of model separation:
1.  **Perfect Model:** 100% separation.
2.  **Realistic Model:** Moderate overlap.
3.  **Random Model:** Complete overlap.

In [ ]:
n_samples = 100
y_true = np.concatenate([np.ones(n_samples), np.zeros(n_samples)]).astype(int)

# Perfect Model
scores_perfect = np.concatenate([
    np.random.normal(0.85, 0.05, n_samples),
    np.random.normal(0.15, 0.05, n_samples)
])

# Realistic Model
scores_real = np.concatenate([
    np.random.normal(0.65, 0.15, n_samples),
    np.random.normal(0.35, 0.15, n_samples)
])

# Random Model
scores_random = np.concatenate([
    np.random.normal(0.5, 0.15, n_samples),
    np.random.normal(0.5, 0.15, n_samples)
])

# Clip scores to [0.0, 1.0]
scores_perfect = np.clip(scores_perfect, 0, 1)
scores_real = np.clip(scores_real, 0, 1)
scores_random = np.clip(scores_random, 0, 1)

Let's plot score histograms for these models.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

models = {
    'Perfect Model': scores_perfect,
    'Realistic Model': scores_real,
    'Random Model': scores_random
}

for idx, (name, scores) in enumerate(models.items()):
    ax = axes[idx]
    ax.hist(scores[y_true == 1], color='blue', alpha=0.5, bins=15, label='Class 1: Positives')
    ax.hist(scores[y_true == 0], color='red', alpha=0.5, bins=15, label='Class 0: Negatives')
    ax.set_title(f"{name}\nROC AUC: {roc_auc_score(y_true, scores):.3f}")
    ax.set_xlabel('Predicted Probability')
    ax.set_ylabel('Frequency')
    ax.legend()
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 2. Implementing AUC Calculation from Scratch

### Method 1: Integration using the Trapezoidal Rule
Once we obtain the ROC points `(fpr, tpr)`, we integrate by calculating the area of adjacent trapezoids:
$$\text{Area} = \sum_{i=1}^{k} \frac{1}{2} (\text{tpr}_i + \text{tpr}_{i-1}) \cdot (\text{fpr}_i - \text{fpr}_{i-1})$$

### Method 2: Pairwise Probability Comparison
We compare all positive samples $x^+$ and negative samples $x^-$. If $f(x^+) > f(x^-)$, we add 1. If they are equal, we add 0.5. The sum divided by $|P| \times |N|$ is the AUC.

In [ ]:
def custom_roc_curve(y_true, scores):
    thresholds = np.sort(scores)[::-1]
    thresholds = np.concatenate([[1.001], thresholds])
    tprs, fprs = [], []
    for thresh in thresholds:
        y_pred = (scores >= thresh).astype(int)
        TP = np.sum((y_true == 1) & (y_pred == 1))
        TN = np.sum((y_true == 0) & (y_pred == 0))
        FP = np.sum((y_true == 0) & (y_pred == 1))
        FN = np.sum((y_true == 1) & (y_pred == 0))
        tprs.append(TP / (TP + FN) if (TP + FN) > 0 else 0.0)
        fprs.append(FP / (TN + FP) if (TN + FP) > 0 else 0.0)
    return np.array(fprs), np.array(tprs)

# Method 1: Trapezoidal Integration
def custom_auc_integration(fpr, tpr):
    """
    Calculate Area Under the Curve using the trapezoidal rule.
    """
    area = 0.0
    for i in range(1, len(fpr)):
        height = tpr[i] + tpr[i-1]
        width = fpr[i] - fpr[i-1]
        area += 0.5 * height * width
    return area

# Method 2: Pairwise Comparisons
def custom_auc_probabilistic(y_true, scores):
    """
    Calculate AUC as the probability of ranking a positive sample higher than a negative sample.
    """
    pos_scores = scores[y_true == 1]
    neg_scores = scores[y_true == 0]
    
    comparisons = 0.0
    for p in pos_scores:
        for n in neg_scores:
            if p > n:
                comparisons += 1.0
            elif p == n:
                comparisons += 0.5
                
    return comparisons / (len(pos_scores) * len(neg_scores))

# Calculate for Realistic Model
fpr, tpr = custom_roc_curve(y_true, scores_real)
auc_int = custom_auc_integration(fpr, tpr)
auc_prob = custom_auc_probabilistic(y_true, scores_real)
auc_sklearn = roc_auc_score(y_true, scores_real)

print(f"Integration Method AUC  : {auc_int:.5f}")
print(f"Probabilistic Method AUC: {auc_prob:.5f}")
print(f"Scikit-Learn AUC        : {auc_sklearn:.5f}")

Both methods yield the exact same value!

## 💡 Connection to Computer Vision & YOLO
*   **AUC under PR Curve:** The term **Average Precision (AP)** is simply the area under the **Precision-Recall Curve** (AUC-PR). AP represents the quality of detections for a single class.
*   **mAP (mean Average Precision):** YOLO takes the AP (AUC-PR) of all 26 classes in your PTT dataset and averages them to compute the overall **mAP** (e.g. `mAP50`). This makes AP/AUC the most important single score when evaluating object detectors!